# 71 — Train multi-modal cross-encoder reranker (Stage B)

Cascade: Stage A multi-modal bi-encoder (top-100) -> this cross-encoder -> top-20.
Reuses Stage A's `TripleJsonlDataset` + modality artifacts. v1 is single-positive
(teacher multi-positive dormant; see the builder's v1 NOTE).

Run order: 1 CONFIG -> 2 setup -> 3 build triples -> 4 train -> 5 dev-eval gate.
GATE before training: Stage A dev nDCG@20 >= 0.16 (run nb 70 Phase 5 first).

In [ ]:
# 1) CONFIG — Stage B multi-modal cross-encoder. Edit, then run cells 2-5.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

BRANCH               = 'stage-b-cross-encoder'
HUB_USER             = 'OrRim123'

# --- Stage A (trained multi-modal bi-encoder) — supplies the top-100 pool ---
# MUST match what nb 70 pushed (its RUN_NAME + '-merged') and its catalog-embed label.
STAGE_A_HUB_REPO     = 'OrRim123/recsys2026-bge-base-en-music-v1-mm-merged'
EMBED_LABEL          = 'bge-base-en-music-v1-mm-merged'

# --- Stage B reranker ---
RERANKER_BASE        = 'BAAI/bge-reranker-v2-m3'
RERANKER_HUB_REPO    = HUB_USER + '/recsys2026-mm-reranker-v1'

# --- Shared Phase 0 artifacts (MUST be the same dir nb 70 used) ---
MULTIMODAL_ARTIFACTS = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/multimodal'
TEACHER_SCORES_PATH  = MULTIMODAL_ARTIFACTS + '/teacher_scores.parquet'  # v1: multi-pos dormant
MULTIPOSITIVE_THRESHOLD = 0.85

# --- Data + training ---
TRIPLES_OUT          = 'experiments/cache/retrieval_v2/triples_reranker_mm.jsonl'
POOL_SIZE            = 100
N_NEGATIVES          = 7
EPOCHS               = 3
LR                   = 2e-5
BATCH_SIZE           = 8       # ~570M backbone, full FT
MAX_LENGTH           = 512
MAX_ROWS             = 0       # 0 = all; set e.g. 200 for a quick smoke run
TRAIN_OUTPUT_DIR     = '/content/mm_reranker_finetune'

# --- Catalog / retriever lookups (MUST match nb 70) ---
ITEM_DB              = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS_TYPES         = 'track_name,artist_name,album_name'
CACHE_DIR            = 'experiments/cache/retrieval_v2'
print('CONFIG set. Stage A:', STAGE_A_HUB_REPO, '-> reranker:', RERANKER_HUB_REPO)

In [ ]:
# 2) Setup — clone branch + HF auth + Drive mount + cache symlink + deps.
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the Drive cache so triples + catalog embeddings persist across sessions.
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
_src = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache'
_dst = LOCAL_BASE + '/retrieval_v2'
os.makedirs(_src, exist_ok=True)
if os.path.islink(_dst): os.unlink(_dst)
elif os.path.exists(_dst):
    import shutil; shutil.rmtree(_dst)
os.symlink(_src, _dst)

!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub'

In [ ]:
# 3) Build Stage B training triples — Stage A top-100 per training query.
# Single-positive v1 (teacher multi-pos dormant; see builder v1 NOTE). Set
# MAX_ROWS>0 in CONFIG for a quick smoke run first.
_maxrows = ('--max-rows ' + str(MAX_ROWS)) if MAX_ROWS else ''
!cd /content/recsys2026 && python -u scripts/build_cross_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --stage-a-hub-repo {STAGE_A_HUB_REPO} \
    --multimodal-artifacts {MULTIMODAL_ARTIFACTS} \
    --teacher-scores-path {TEACHER_SCORES_PATH} \
    --multipositive-threshold {MULTIPOSITIVE_THRESHOLD} \
    --output {TRIPLES_OUT} --pool-size {POOL_SIZE} \
    --embed-label {EMBED_LABEL} --cache-dir {CACHE_DIR} \
    --history-corpus-types {CORPUS_TYPES} {_maxrows} \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/ce_build_log.txt
!wc -l {TRIPLES_OUT}

In [ ]:
# 4) Train the multi-modal cross-encoder (full FT, bf16). Pushes merged -> Hub.
!cd /content/recsys2026 && python -u scripts/train_cross_encoder.py \
    --triples {TRIPLES_OUT} \
    --multimodal-artifacts {MULTIMODAL_ARTIFACTS} \
    --base-model {RERANKER_BASE} \
    --output-dir {TRAIN_OUTPUT_DIR} \
    --hub-repo {RERANKER_HUB_REPO} \
    --epochs {EPOCHS} --lr {LR} --batch-size {BATCH_SIZE} \
    --n-negatives {N_NEGATIVES} --max-length {MAX_LENGTH} \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/ce_train_log.txt

In [ ]:
# 5) Dev eval — Stage A vs Stage A+B cascade nDCG@20.
# GATE (plan Phase 9): Stage B must add >= +0.04 nDCG@20 over Stage A alone.
import sys, math
import numpy as np
from datasets import load_dataset
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.retrieval_modules.dense_multimodal_local import DENSE_MULTIMODAL_LOCAL
from mcrs.rerankers.multimodal_cross_encoder_rerank import MULTIMODAL_RERANKER
from mcrs.retrieval_modules.bge_m3_format import format_query_text
from build_bi_encoder_training_data import _iter_conversation_turns

CORPUS = CORPUS_TYPES.split(',')
retriever = DENSE_MULTIMODAL_LOCAL(
    dataset_name=ITEM_DB, split_types=[CORPUS_TYPES], corpus_types=CORPUS,
    cache_dir=CACHE_DIR, model_dir=STAGE_A_HUB_REPO, embed_label=EMBED_LABEL,
    multimodal_artifacts=MULTIMODAL_ARTIFACTS,
)
reranker = MULTIMODAL_RERANKER(
    model_dir=RERANKER_HUB_REPO, multimodal_artifacts=MULTIMODAL_ARTIFACTS,
    item_db_name=ITEM_DB, track_split_types=['all_tracks'], corpus_types=CORPUS,
    cache_dir=CACHE_DIR,
)

dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
rows = _iter_conversation_turns(dev)[:500]
queries = [format_query_text(r.get('chat_history') or [], r.get('current_user_query',''),
           r.get('user_profile_raw'), r.get('conversation_goal'), mode='bge_m3_structured')
           for r in rows]
user_ids = [r.get('user_id') for r in rows]
golds = [r['track_id'] for r in rows]

cand100 = retriever.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids)

def ndcg20(ranked, gold):
    top = ranked[:20]
    return 1.0 / math.log2(top.index(gold) + 2) if gold in top else 0.0

stage_a = np.mean([ndcg20(c, g) for c, g in zip(cand100, golds)])
reranked = reranker.rerank(queries, cand100, topk=20, user_ids=user_ids)
stage_ab = np.mean([ndcg20(r, g) for r, g in zip(reranked, golds)])
print(f'Stage A   nDCG@20: {stage_a:.4f}')
print(f'Stage A+B nDCG@20: {stage_ab:.4f}  (lift {stage_ab - stage_a:+.4f}; gate >= +0.04)')
print('PASS' if stage_ab - stage_a >= 0.04 else 'CHECK — Stage B lift below the +0.04 gate')

## Deploy the cascade (Blind-A / dev YAML)

The cascade is config, not new code (the reranker is registered in
`mcrs/rerankers`). In the inference YAML (nb 73 / run_inference_*), set:

```yaml
retrieval_type: wrrf_bm25_multimodal_v1
retrieval_topk: 100                       # feed the reranker a top-100 pool
reranker_type: multimodal_cross_encoder
reranker_model_path: OrRim123/recsys2026-mm-reranker-v1
reranker_multimodal_artifacts: /content/drive/MyDrive/recsys2026_retrieval_v2_cache/multimodal
```

`user_ids` flow to the reranker automatically via `batch_chat`.